In [7]:
import os
from importlib import resources
from pathlib import Path
from typing import Dict

SHADER_PACKAGE = "wxx.shaders"

def _get_shaders_root():
    """Returns the Traversable root node for the shaders package."""
    return resources.files(SHADER_PACKAGE)

def walk_shader_files(traversable_dir=None, current_rel_path="") -> Dict[str, str]:
    """Recursively collects all .wgsl files into a dict keyed by relative path."""
    if traversable_dir is None:
        traversable_dir = _get_shaders_root()

    shader_files = {}
    for item in traversable_dir.iterdir():
        rel_path = f"{current_rel_path}/{item.name}" if current_rel_path else item.name
        
        if item.is_dir():
            shader_files.update(walk_shader_files(item, rel_path))
        elif item.name.endswith(".wgsl"):
            shader_files[rel_path] = item.read_text(encoding="utf-8")

    return shader_files

walk_shader_files(Path("."))

{'activations/relu.wgsl': '@group(0) @binding(0) var<storage, read> dataA : array<f32>;\n@group(0) @binding(1) var<storage, read_write> dataOut : array<f32>;\n\nconst WORKGROUP_SIZE : u32 = 256u;\n\n@compute @workgroup_size(256)\nfn main(\n    @builtin(global_invocation_id) global_id : vec3<u32>,\n    @builtin(num_workgroups) num_workgroups : vec3<u32>\n) {\n    let total_elements = arrayLength(&dataOut);\n    let stride = num_workgroups.x * WORKGROUP_SIZE;\n    \n    var i = global_id.x;\n    while (i < total_elements) {\n        dataOut[i] = max(0.0, dataA[i]);\n        i += stride;\n    }\n}',
 'activations/relubackwards.wgsl': '@group(0) @binding(0) var<storage, read> dataA : array<f32>;\n@group(0) @binding(1) var<storage, read_write> dataOut : array<f32>;\n\nconst WORKGROUP_SIZE : u32 = 256u;\n\n@compute @workgroup_size(256)\nfn main(\n    @builtin(global_invocation_id) global_id : vec3<u32>,\n    @builtin(num_workgroups) num_workgroups : vec3<u32>\n) {\n    let total_elements = a